# 30. The Yard Pre-Marshalling for Exports Problem
## Tier 2: List Scheduling Algorithm

### Goal
Implement a list scheduling heuristic that treats containers as jobs to be scheduled onto stack positions, prioritizing moves that create maximum immediate improvement while maintaining feasibility constraints.

### Key Assumptions
- Only top containers in each stack can be moved (accessibility constraint)
- Priority queue drives move selection based on composite scoring
- Move evaluation considers priority violations, stack utilization, and efficiency
- Algorithm runs in polynomial time for practical problem sizes

### Approach (Step-by-Step)
1. **Initial Analysis**: Identify moveable containers and priority violations
2. **Priority Queue**: Create scoring function for container move evaluation
3. **Move Selection**: Iteratively select highest-scoring valid moves
4. **Constraint Handling**: Ensure all moves respect physical and operational constraints
5. **Termination**: Stop when all priority violations are eliminated or no moves possible

### What to Look for in the Results
- Number of moves required to eliminate violations
- Algorithm execution time and scalability
- Solution quality compared to optimal solution
- Priority queue scoring effectiveness

### Concrete Example
Bay with 3 stacks and initial configuration:
- Stack 0: [Container_E(5), Container_B(2)]
- Stack 1: [Container_A(1), Container_D(4)]
- Stack 2: [Container_C(3)]

In [1]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import heapq
from collections import defaultdict
import time

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """Container with priority and position information"""
    id: str
    priority: int  # Lower values = higher priority
    current_stack: int
    current_tier: int
    
    def __lt__(self, other):
        """For priority queue ordering"""
        return self.priority < other.priority

@dataclass
class Move:
    """Container move operation"""
    container_id: str
    from_stack: int
    from_tier: int
    to_stack: int
    to_tier: int
    score: float

class ListScheduler:
    """List scheduling algorithm for pre-marshalling"""
    
    def __init__(self, stacks: int, tiers: int):
        self.stacks = stacks
        self.tiers = tiers
        self.move_history = []
        self.execution_log = []
        
    def get_stack_height(self, containers: List[Container], stack: int) -> int:
        """Get current height of a stack"""
        stack_containers = [c for c in containers if c.current_stack == stack]
        return len(stack_containers)
    
    def get_top_container(self, containers: List[Container], stack: int) -> Optional[Container]:
        """Get the top container in a stack"""
        stack_containers = [c for c in containers if c.current_stack == stack]
        if not stack_containers:
            return None
        # Return container with highest tier (topmost)
        return max(stack_containers, key=lambda c: c.current_tier)
    
    def count_priority_violations(self, containers: List[Container]) -> int:
        """Count priority violations in current configuration"""
        violations = 0
        
        # Create grid representation
        grid = [[None for _ in range(self.tiers)] for _ in range(self.stacks)]
        for container in containers:
            if container.current_tier < self.tiers:
                grid[container.current_stack][container.current_tier] = container.priority
        
        # Check each stack for priority violations
        for stack in range(self.stacks):
            for tier in range(self.tiers - 1):
                lower_container = grid[stack][tier]
                upper_container = grid[stack][tier + 1]
                
                if lower_container is not None and upper_container is not None:
                    # Violation if lower priority (higher number) is below higher priority (lower number)
                    if lower_container > upper_container:
                        violations += 1
                        
        return violations

In [ ]:
    def calculate_move_score(self, containers: List[Container], move: Move) -> float:
        """Calculate priority score for a potential move"""
        score = 0.0
        
        # Component 1: Priority violation reduction (weight: 50)
        container = next(c for c in containers if c.id == move.container_id)
        
        # Check if move resolves violations in current stack
        current_stack_containers = [c for c in containers if c.current_stack == move.from_stack and c.id != move.container_id]
        current_stack_containers.sort(key=lambda c: c.current_tier)
        
        violations_before = 0
        violations_after = 0
        
        # Count violations involving this container in current position
        for other in current_stack_containers:
            if other.current_tier > container.current_tier:  # Container is below other
                if container.priority > other.priority:
                    violations_before += 1
            elif other.current_tier < container.current_tier:  # Container is above other
                if other.priority > container.priority:
                    violations_before += 1
        
        # Estimate violations after move (simplified - assume good placement)
        violations_after = max(0, violations_before - 1)
        violation_reduction = violations_before - violations_after
        score += 50 * violation_reduction
        
        # Component 2: Stack balance improvement (weight: 20)
        from_height = self.get_stack_height(containers, move.from_stack)
        to_height = self.get_stack_height(containers, move.to_stack)
        avg_height = sum(self.get_stack_height(containers, s) for s in range(self.stacks)) / self.stacks
        
        # Prefer moves that balance stack heights
        balance_before = abs(from_height - avg_height) + abs(to_height - avg_height)
        balance_after = abs(from_height - 1 - avg_height) + abs(to_height + 1 - avg_height)
        balance_improvement = balance_before - balance_after
        score += 20 * balance_improvement
        
        # Component 3: Move efficiency (weight: 15)
        # Prefer shorter moves and lower tier moves
        distance = abs(move.from_stack - move.to_stack)
        tier_cost = move.from_tier  # Higher tier = easier to move
        efficiency = 15 - (2 * distance) - tier_cost
        score += max(0, efficiency)
        
        # Component 4: Priority-based bonus (weight: 15)
        # Higher priority containers get bonus for good placement
        priority_bonus = 15 * (10 - container.priority) / 10  # Inverse priority
        score += priority_bonus
        
        return score

In [ ]:
    def generate_valid_moves(self, containers: List[Container]) -> List[Move]:
        """Generate all valid moves for current configuration"""
        moves = []
        
        for stack in range(self.stacks):
            top_container = self.get_top_container(containers, stack)
            if top_container is None:
                continue
                
            # Try moving to all other stacks
            for target_stack in range(self.stacks):
                if target_stack == stack:
                    continue
                    
                # Check if target stack has space
                target_height = self.get_stack_height(containers, target_stack)
                if target_height < self.tiers:
                    # Create move
                    move = Move(
                        container_id=top_container.id,
                        from_stack=stack,
                        from_tier=top_container.current_tier,
                        to_stack=target_stack,
                        to_tier=target_height,
                        score=0.0  # Will be calculated later
                    )
                    
                    # Calculate move score
                    move.score = self.calculate_move_score(containers, move)
                    moves.append(move)
                    
        return moves
    
    def execute_move(self, containers: List[Container], move: Move) -> List[Container]:
        """Execute a move and return updated container list"""
        new_containers = containers.copy()
        
        # Find and update the moved container
        for container in new_containers:
            if container.id == move.container_id:
                container.current_stack = move.to_stack
                container.current_tier = move.to_tier
                break
                
        return new_containers
    
    def solve(self, initial_containers: List[Container], max_moves: int = 100) -> Tuple[List[Container], List[Move]]:
        """Solve pre-marshalling using list scheduling algorithm"""
        containers = initial_containers.copy()
        self.move_history = []
        self.execution_log = []
        
        iteration = 0
        while iteration < max_moves:
            # Check if solution is optimal
            violations = self.count_priority_violations(containers)
            if violations == 0:
                self.execution_log.append(f"Iteration {iteration}: Solution optimal - 0 violations")
                break
                
            # Generate valid moves
            valid_moves = self.generate_valid_moves(containers)
            
            if not valid_moves:
                self.execution_log.append(f"Iteration {iteration}: No valid moves available")
                break
                
            # Select best move
            best_move = max(valid_moves, key=lambda m: m.score)
            
            # Execute move
            containers = self.execute_move(containers, best_move)
            self.move_history.append(best_move)
            
            # Log progress
            new_violations = self.count_priority_violations(containers)
            self.execution_log.append(
                f"Iteration {iteration}: Move {best_move.container_id} from stack {best_move.from_stack} to {best_move.to_stack}, "
                f"score={best_move.score:.1f}, violations: {violations} -> {new_violations}"
            )
            
            iteration += 1
            
        return containers, self.move_history

In [ ]:
def visualize_bay_state(containers: List[Container], stacks: int, tiers: int, title: str, ax=None):
    """Visualize container bay configuration"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create grid
    grid = [[None for _ in range(tiers)] for _ in range(stacks)]
    for container in containers:
        if container.current_tier < tiers:
            grid[container.current_stack][container.current_tier] = container
    
    # Draw containers
    for stack in range(stacks):
        for tier in range(tiers):
            container = grid[stack][tier]
            if container:
                # Color based on priority
                max_priority = max(c.priority for c in containers)
                color_intensity = container.priority / max_priority
                color = plt.cm.RdYlBu_r(color_intensity)
                
                # Draw container rectangle
                rect = plt.Rectangle((stack * 1.2, tier * 0.8), 1.0, 0.6, 
                                  facecolor=color, edgecolor='black', linewidth=2)
                ax.add_patch(rect)
                
                # Add container label
                ax.text(stack * 1.2 + 0.5, tier * 0.8 + 0.3, f"{container.id}\n({container.priority})", 
                       ha='center', va='center', fontweight='bold', fontsize=10)
            else:
                # Draw empty position
                rect = plt.Rectangle((stack * 1.2, tier * 0.8), 1.0, 0.6, 
                                  facecolor='lightgray', edgecolor='gray', linewidth=1, alpha=0.3)
                ax.add_patch(rect)
    
    ax.set_xlim(-0.5, stacks * 1.2)
    ax.set_ylim(-0.5, tiers * 0.8)
    ax.set_xlabel('Stack', fontsize=12)
    ax.set_ylabel('Tier', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    # Add stack and tier labels
    for stack in range(stacks):
        ax.text(stack * 1.2 + 0.5, -0.3, f'S{stack}', ha='center', fontweight='bold')
    for tier in range(tiers):
        ax.text(-0.3, tier * 0.8 + 0.3, f'T{tier}', ha='center', fontweight='bold')

def visualize_algorithm_progression(initial_containers: List[Container], 
                                   final_containers: List[Container],
                                   move_history: List[Move],
                                   stacks: int, tiers: int):
    """Visualize the progression of the algorithm"""
    n_moves = len(move_history)
    if n_moves == 0:
        return
        
    # Create subplots for key stages
    n_stages = min(5, n_moves + 1)  # Initial + up to 4 intermediate stages
    fig, axes = plt.subplots(1, n_stages, figsize=(4 * n_stages, 6))
    if n_stages == 1:
        axes = [axes]
    
    # Show initial state
    visualize_bay_state(initial_containers, stacks, tiers, 'Initial State', axes[0])
    
    # Show intermediate states
    if n_stages > 1:
        step_size = max(1, n_moves // (n_stages - 1))
        current_containers = initial_containers.copy()
        
        for i in range(1, n_stages):
            move_idx = min(i * step_size, n_moves - 1)
            
            # Apply moves up to this point
            for j in range(move_idx + 1):
                move = move_history[j]
                for container in current_containers:
                    if container.id == move.container_id:
                        container.current_stack = move.to_stack
                        container.current_tier = move.to_tier
                        break
            
            title = f'After {move_idx + 1} Moves'
            if i == n_stages - 1:
                title = f'Final State ({n_moves} moves)'
            visualize_bay_state(current_containers, stacks, tiers, title, axes[i])
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Create the concrete example from the problem statement
print("=" * 60)
print("YARD PRE-MARSHALLING PROBLEM - LIST SCHEDULING ALGORITHM")
print("=" * 60)

# Define containers from the concrete example
containers = [
    Container('E', 5, 0, 1),  # Stack 0, Tier 1, Priority 5 (top)
    Container('B', 2, 0, 0),  # Stack 0, Tier 0, Priority 2 (bottom)
    Container('A', 1, 1, 1),  # Stack 1, Tier 1, Priority 1 (top)
    Container('D', 4, 1, 0),  # Stack 1, Tier 0, Priority 4 (bottom)
    Container('C', 3, 2, 0)   # Stack 2, Tier 0, Priority 3
]

stacks, tiers = 3, 4

print(f"\nProblem Configuration:")
print(f"- Stacks: {stacks}")
print(f"- Tiers: {tiers}")
print(f"- Containers: {len(containers)}")

print(f"\nInitial Container Configuration:")
for container in containers:
    print(f"  Container {container.id}: Priority {container.priority}, Stack {container.current_stack}, Tier {container.current_tier}")

# Check initial violations
initial_violations = ListScheduler(stacks, tiers).count_priority_violations(containers)
print(f"\nInitial Priority Violations: {initial_violations}")

# Visualize initial configuration
visualize_bay_state(containers, stacks, tiers, 'Initial Bay Configuration')

In [ ]:
# Initialize and run the list scheduling algorithm
scheduler = ListScheduler(stacks, tiers)

print("\n" + "=" * 60)
print("LIST SCHEDULING ALGORITHM EXECUTION")
print("=" * 60)

# Time the execution
start_time = time.time()
final_containers, move_history = scheduler.solve(containers, max_moves=50)
execution_time = time.time() - start_time

print(f"\nAlgorithm completed in {execution_time:.4f} seconds")
print(f"Total moves executed: {len(move_history)}")

# Print execution log
print(f"\nExecution Log:")
for log_entry in scheduler.execution_log:
    print(f"  {log_entry}")

In [ ]:
# Display final results
print("\n" + "=" * 60)
print("FINAL SOLUTION ANALYSIS")
print("=" * 60)

print(f"\nFinal Container Configuration:")
for container in final_containers:
    print(f"  Container {container.id}: Priority {container.priority}, Stack {container.current_stack}, Tier {container.current_tier}")

# Check solution quality
final_violations = scheduler.count_priority_violations(final_containers)

print(f"\nSolution Quality:")
print(f"  Initial priority violations: {initial_violations}")
print(f"  Final priority violations: {final_violations}")

if final_violations == 0:
    print("  ✓ Perfect solution achieved - all violations eliminated!")
else:
    print(f"  ⚠ {final_violations} violations remain")

# Visualize algorithm progression
visualize_algorithm_progression(containers, final_containers, move_history, stacks, tiers)

### Why This Tier Exists vs Network Flow Formulation

**List Scheduling Algorithm Advantages:**
- **Computational Efficiency**: Polynomial time complexity O(n²·s·t) vs exponential for exact methods
- **Real-time Applicability**: Suitable for operational decision-making with time constraints
- **Scalability**: Can handle larger problem instances (50+ containers) practically
- **Flexibility**: Easy to modify scoring function for different objectives

**Limitations of Network Flow (Tier 1):**
- **Computational Complexity**: Exponential scaling limits practical applicability
- **Real-time Constraints**: Not suitable for dynamic operational environments
- **Implementation Complexity**: Requires specialized optimization solvers for large instances

### Pros and Cons vs Alternative Approaches

**Pros:**
- ✓ Fast execution suitable for real-time operations
- ✓ Handles medium to large problem instances efficiently
- Transparent decision-making process through scoring function
- Easy to customize for different operational priorities
- Guaranteed improvement over initial configuration

**Cons:**
- ✗ No optimality guarantee - may not find best possible solution
- ✗ Solution quality depends on scoring function design
- ✗ May get stuck in local optima
- ✗ Performance varies with problem structure
- ✗ Requires careful tuning of scoring weights

### When to Use This Tier

**Use List Scheduling when:**
- Real-time decision making is required (seconds to minutes)
- Problem instances are medium to large (20-100 containers)
- Approximate solutions are acceptable for operational use
- Computational resources are limited
- Dynamic environments require fast re-planning
- Integration with terminal operating systems is needed

**Consider alternatives when:**
- Optimal solution is required for benchmarking
- Problem instances are small enough for exact methods
- Solution optimality is more critical than execution time
- Academic research or theoretical analysis is needed